# 🫁 Audio-Based Respiratory Biomarker Analysis
### Cough & Breathing Detection with Signal Processing + Machine Learning

**What this notebook does**
1. Loads audio (your own upload, or a built-in synthetic recording with known ground truth)
2. Preprocesses it (resample, band-pass filter, normalise)
3. Extracts acoustic features (MFCC, spectral, temporal)
4. Trains a classifier to label 1-second windows as **cough / breathing / background**
5. Detects cough events in a long recording and counts them
6. Computes respiratory biomarkers: **cough count & rate, cough duration, intensity, inter-cough interval, breathing rate**
7. Visualises everything and exports a CSV report

> ⚠️ **Research / educational use only.** This is not a medical device and must not be used for diagnosis.

**How to run:** `Runtime → Run all`. Everything works with no uploads (synthetic data + optional real ESC-50 clips).

## 1. Setup

In [ ]:
!pip install -q librosa soundfile scikit-learn seaborn

In [ ]:
import os, io, warnings, urllib.request
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa, librosa.display
import soundfile as sf
import scipy.signal as sps
from scipy.stats import kurtosis
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from IPython.display import Audio, display

warnings.filterwarnings("ignore")
rng = np.random.default_rng(42)
plt.rcParams["figure.dpi"] = 110

# ----------------- CONFIG -----------------
SR          = 16000      # working sample rate (Hz)
WIN_S       = 1.0        # classification window length (s)
HOP_S       = 0.25       # sliding-window hop for inference (s)
CLASSES     = ["background", "breath", "cough"]
USE_ESC50   = True       # also train on real cough/breathing clips from ESC-50 (needs internet)
N_SYNTH     = 400        # synthetic windows per class
# ------------------------------------------
print("Setup OK | librosa", librosa.__version__)

## 2. Audio utilities & synthetic data generator
Real, labelled respiratory audio is often restricted, so we include a **synthetic generator**
(coughs = explosive burst + voiced tail, breaths = slow noise swells, background = noise / hum / speech-like).
This gives us **ground truth** to validate the pipeline. Later you can swap in real recordings.

In [ ]:
def bandpass(x, sr, lo=50, hi=4000, order=4):
    sos = sps.butter(order, [lo, hi], btype="band", fs=sr, output="sos")
    return sps.sosfiltfilt(sos, x)

def norm(x):
    return x / (np.max(np.abs(x)) + 1e-9)

def colored_noise(n, beta=1.0):
    """1/f^beta noise (beta=0 white, 1 pink, 2 brown)."""
    X = np.fft.rfft(rng.standard_normal(n))
    f = np.fft.rfftfreq(n); f[0] = f[1]
    X = X / f ** (beta / 2)
    return norm(np.fft.irfft(X, n))

def synth_cough(sr=SR):
    dur = rng.uniform(0.25, 0.5); n = int(dur * sr); t = np.arange(n) / sr
    # explosive phase + decaying tail
    env = np.exp(-t / rng.uniform(0.06, 0.12))
    env[: int(0.008 * sr)] *= np.linspace(0, 1, int(0.008 * sr))
    burst = bandpass(rng.standard_normal(n), sr, 300, 5000) * env
    # voiced phase: harmonic tone with falling pitch
    f0 = rng.uniform(150, 260) * (1 - 0.3 * t / dur)
    voiced = sum(np.sin(2 * np.pi * np.cumsum(f0 * k) / sr) / k for k in range(1, 6))
    venv = np.exp(-((t - 0.6 * dur) ** 2) / (2 * (0.1 * dur) ** 2)) * rng.uniform(0.1, 0.4)
    x = burst + voiced * venv
    if rng.random() < 0.35:  # double-burst cough
        d = int(rng.uniform(0.12, 0.2) * sr)
        x = np.concatenate([x, np.zeros(d)])
        x[d:d + n] += 0.6 * x[:n]
    return norm(x)

def synth_breath(sr=SR, dur=None):
    dur = dur or rng.uniform(2.0, 4.0); n = int(dur * sr); t = np.arange(n) / sr
    ins = np.sin(np.pi * np.clip(t / (0.42 * dur), 0, 1)) ** 2 * (t < 0.42 * dur)
    exp_ = np.sin(np.pi * np.clip((t - 0.48 * dur) / (0.45 * dur), 0, 1)) ** 2 * (t > 0.48 * dur) * 0.8
    noise = bandpass(colored_noise(n, 0.7), sr, 200, 2000)
    return norm(noise * (ins + exp_))

def synth_background(n, sr=SR):
    kind = rng.choice(["white", "pink", "hum", "speech"])
    if kind == "white":  x = rng.standard_normal(n)
    elif kind == "pink": x = colored_noise(n, 1.2)
    elif kind == "hum":
        t = np.arange(n) / sr
        x = np.sin(2 * np.pi * 50 * t) + 0.5 * np.sin(2 * np.pi * 100 * t) + 0.2 * rng.standard_normal(n)
    else:  # speech-like: pitch-modulated harmonics with syllabic AM
        t = np.arange(n) / sr
        f0 = 120 + 30 * np.sin(2 * np.pi * 0.7 * t)
        x = sum(np.sin(2 * np.pi * np.cumsum(f0 * k) / sr) / k for k in range(1, 12))
        x *= (0.5 + 0.5 * np.sin(2 * np.pi * rng.uniform(2.5, 4.5) * t)) ** 2
        x = bandpass(x, sr, 100, 3500)
    return norm(x)

def add_noise(x, snr_db):
    p = np.mean(x ** 2) + 1e-12
    noise = colored_noise(len(x), rng.uniform(0, 1.5))
    noise *= np.sqrt(p / 10 ** (snr_db / 10) / (np.mean(noise ** 2) + 1e-12))
    return x + noise

def make_window(label, sr=SR, win_s=WIN_S):
    n = int(win_s * sr)
    bg_level = rng.uniform(0.005, 0.05)
    bg = colored_noise(n, rng.uniform(0, 1.5)) * bg_level
    if label == "background":
        x = synth_background(n, sr) * rng.uniform(0.02, 0.4) + bg
    elif label == "breath":
        b = synth_breath(sr, dur=rng.uniform(1.5, 3))
        seg = np.zeros(n); off = rng.integers(-len(b) // 2, n - len(b) // 2)
        for i in range(len(b)):
            if 0 <= i + off < n: seg[i + off] = b[i]
        x = seg * rng.uniform(0.05, 0.4) + bg
    else:
        c = synth_cough(sr); seg = np.zeros(n)
        off = rng.integers(0, max(1, n - len(c))); seg[off:off + min(len(c), n - off)] = c[: n - off]
        x = seg * rng.uniform(0.2, 0.9) + bg
    return x.astype(np.float32)

def make_long_recording(duration_s=90, breath_bpm=16, n_coughs=9, sr=SR, seed=7):
    """Synthetic 'overnight-style' clip with known cough times + breathing rate."""
    global rng
    old = rng; rng = np.random.default_rng(seed)
    n = int(duration_s * sr); x = colored_noise(n, 1.0) * 0.02
    period = 60.0 / breath_bpm; t0 = rng.uniform(0, 1)
    while t0 < duration_s - period:
        b = synth_breath(sr, dur=period * rng.uniform(0.9, 1.0)) * rng.uniform(0.12, 0.2)
        i = int(t0 * sr); x[i:i + len(b)] += b[: n - i]; t0 += period
    starts = np.sort(rng.uniform(5, duration_s - 5, n_coughs))
    while np.min(np.diff(starts)) < 3: starts = np.sort(rng.uniform(5, duration_s - 5, n_coughs))
    truth = []
    for s in starts:
        c = synth_cough(sr) * rng.uniform(0.4, 0.9); i = int(s * sr)
        x[i:i + len(c)] += c[: n - i]; truth.append((s, s + len(c) / sr))
    rng = old
    return x.astype(np.float32), truth

# quick listen
demo = {k: make_window(k) for k in CLASSES}
fig, ax = plt.subplots(1, 3, figsize=(13, 2.6))
for a, (k, w) in zip(ax, demo.items()):
    a.plot(np.arange(len(w)) / SR, w, lw=.6); a.set_title(k); a.set_xlabel("s")
plt.tight_layout(); plt.show()
display(Audio(demo["cough"], rate=SR)); display(Audio(demo["breath"], rate=SR))

## 3. Feature extraction
For each 1-s window we compute a compact feature vector:
- **MFCC (20)** mean/std + delta-MFCC mean – timbre of the sound
- **Spectral**: centroid, bandwidth, rolloff, flatness, contrast
- **Temporal**: zero-crossing rate, RMS energy (mean/std/max), crest factor, kurtosis (cough = very impulsive)
- **Band energy ratios**: low (<500 Hz), mid (0.5–2 kHz), high (>2 kHz)

In [ ]:
def extract_features(y, sr=SR):
    y = np.asarray(y, dtype=np.float32)
    y = librosa.effects.preemphasis(y)
    S = np.abs(librosa.stft(y, n_fft=512, hop_length=128)) + 1e-9
    mfcc = librosa.feature.mfcc(S=librosa.power_to_db(librosa.feature.melspectrogram(S=S**2, sr=sr, n_mels=40)), n_mfcc=20)
    d_mfcc = librosa.feature.delta(mfcc)
    cent = librosa.feature.spectral_centroid(S=S, sr=sr)
    bw   = librosa.feature.spectral_bandwidth(S=S, sr=sr)
    roll = librosa.feature.spectral_rolloff(S=S, sr=sr)
    flat = librosa.feature.spectral_flatness(S=S)
    con  = librosa.feature.spectral_contrast(S=S, sr=sr, n_bands=4, fmin=100)
    zcr  = librosa.feature.zero_crossing_rate(y, frame_length=512, hop_length=128)
    rms  = librosa.feature.rms(S=S, frame_length=512)
    freqs = librosa.fft_frequencies(sr=sr, n_fft=512); P = S ** 2
    tot = P.sum() + 1e-12
    lo, mid, hi = (P[freqs < 500].sum() / tot, P[(freqs >= 500) & (freqs < 2000)].sum() / tot, P[freqs >= 2000].sum() / tot)
    feats = np.concatenate([
        mfcc.mean(1), mfcc.std(1), d_mfcc.mean(1),
        [cent.mean(), cent.std(), bw.mean(), roll.mean(), flat.mean(), flat.std()],
        con.mean(1),
        [zcr.mean(), zcr.std(), rms.mean(), rms.std(), rms.max(),
         rms.max() / (rms.mean() + 1e-9), kurtosis(y), lo, mid, hi],
    ])
    return feats

print("Feature vector length:", len(extract_features(demo['cough'])))

## 4. Build the training dataset
Synthetic windows are always used. If `USE_ESC50=True`, we also download a handful of **real** *coughing* and *breathing*
clips (plus background sounds) from the public [ESC-50](https://github.com/karoldvl/ESC-50) dataset to make the model more realistic.

In [ ]:
X, y, src = [], [], []

# ---- synthetic ----
for lab in CLASSES:
    for _ in range(N_SYNTH):
        w = make_window(lab)
        X.append(extract_features(w)); y.append(lab); src.append("synthetic")
print("Synthetic windows:", len(X))

# ---- optional real data: ESC-50 ----
def load_esc50(n_per_class=40):
    base = "https://raw.githubusercontent.com/karoldvl/ESC-50/master/"
    meta = pd.read_csv(base + "meta/esc50.csv")
    mapping = {"coughing": "cough", "breathing": "breath",
               "rain": "background", "wind": "background", "keyboard_typing": "background",
               "clock_tick": "background", "vacuum_cleaner": "background", "washing_machine": "background"}
    Xr, yr = [], []
    os.makedirs("esc50", exist_ok=True)
    for cat, lab in mapping.items():
        files = meta[meta.category == cat].filename.tolist()[: n_per_class if lab != "background" else 12]
        for f in files:
            p = f"esc50/{f}"
            if not os.path.exists(p): urllib.request.urlretrieve(base + "audio/" + f, p)
            a, _ = librosa.load(p, sr=SR)
            a = a[a != 0] if np.any(a != 0) else a                    # ESC-50 pads with digital silence
            n = int(WIN_S * SR)
            if len(a) < n: a = np.pad(a, (0, n - len(a)))
            hops = list(range(0, len(a) - n + 1, n // 2))
            wins = [a[h:h + n] for h in hops]
            if lab in ("cough", "breath"):                            # keep most energetic windows (the event)
                wins = sorted(wins, key=lambda w: -np.sum(w ** 2))[:2]
            else:
                wins = wins[:2]
            for w in wins:
                Xr.append(extract_features(w)); yr.append(lab)
    return Xr, yr

if USE_ESC50:
    try:
        Xr, yr = load_esc50()
        X += Xr; y += yr; src += ["esc50"] * len(yr)
        print("Real ESC-50 windows added:", len(yr), pd.Series(yr).value_counts().to_dict())
    except Exception as e:
        print("⚠️ Could not load ESC-50 (offline?). Continuing with synthetic only.\n", e)

X = np.array(X); y = np.array(y); src = np.array(src)
print("Total:", X.shape, pd.Series(y).value_counts().to_dict())

## 5. Train & evaluate the classifier

In [ ]:
Xtr, Xte, ytr, yte, str_, ste = train_test_split(X, y, src, test_size=0.25, stratify=y, random_state=0)

clf = make_pipeline(StandardScaler(),
                    RandomForestClassifier(n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=0))
clf.fit(Xtr, ytr)
pred = clf.predict(Xte)

print(classification_report(yte, pred, digits=3))
cm = confusion_matrix(yte, pred, labels=CLASSES)
plt.figure(figsize=(4.2, 3.6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Window-level confusion matrix"); plt.show()

if (ste == "esc50").any():
    m = ste == "esc50"
    print("Accuracy on REAL ESC-50 test windows only: %.3f  (n=%d)" % ((pred[m] == yte[m]).mean(), m.sum()))

# Feature importance (top 12)
rf = clf.named_steps["randomforestclassifier"]
imp = pd.Series(rf.feature_importances_).sort_values(ascending=False).head(12)
imp.plot.barh(figsize=(5, 3.5)).invert_yaxis(); plt.title("Top feature indices"); plt.show()

## 6. Load a recording to analyse
- **Option A (default):** synthetic 90-s recording with known coughs and a 16 breaths/min background rhythm.
- **Option B:** upload your own `.wav / .mp3 / .flac / .m4a` file — set `USE_UPLOAD = True`.

In [ ]:
USE_UPLOAD = False   # <- set True to upload your own file

truth = None
if USE_UPLOAD:
    from google.colab import files
    up = files.upload()
    fname = list(up.keys())[0]
    audio, _ = librosa.load(fname, sr=SR, mono=True)
else:
    audio, truth = make_long_recording(duration_s=90, breath_bpm=16, n_coughs=9)
    print("Ground-truth cough times (s):", [round(a, 1) for a, _ in truth], "| true breathing rate: 16 bpm")

# preprocessing
audio = bandpass(audio, SR, 50, 4000)
audio = audio / (np.max(np.abs(audio)) + 1e-9) * 0.9   # peak-normalise (max = -0.9 dBFS)
DUR = len(audio) / SR
print(f"Duration: {DUR:.1f} s")
display(Audio(audio, rate=SR))

## 7. Sliding-window inference

In [ ]:
n, h = int(WIN_S * SR), int(HOP_S * SR)
starts = np.arange(0, len(audio) - n + 1, h)
F = np.array([extract_features(audio[s:s + n]) for s in starts])
proba = clf.predict_proba(F)
classes = list(clf.classes_)
centers = (starts + n / 2) / SR
P = pd.DataFrame(proba, columns=classes); P["t"] = centers

fig, ax = plt.subplots(2, 1, figsize=(13, 5), sharex=True)
ax[0].plot(np.arange(len(audio)) / SR, audio, lw=.5, color="gray"); ax[0].set_ylabel("Amplitude")
if truth:
    for a, b in truth: ax[0].axvspan(a, b, color="red", alpha=.25)
for c, col in zip(classes, ["tab:gray", "tab:blue", "tab:red"]):
    ax[1].plot(P.t, P[c], label=c, color=col)
ax[1].set_ylabel("P(class)"); ax[1].set_xlabel("Time (s)"); ax[1].legend(ncol=3, loc="upper right")
ax[0].set_title("Waveform (red = ground-truth coughs) and class probabilities"); plt.tight_layout(); plt.show()

## 8. Cough event detection
Window probabilities → threshold → merge → refine onset/offset using the short-time energy envelope.

In [ ]:
COUGH_THR   = 0.5    # probability threshold
MIN_GAP_S   = 0.5    # merge detections closer than this
MIN_DUR_S   = 0.08

def detect_coughs(audio, P, thr=COUGH_THR):
    active = P["cough"].values > thr
    t = P["t"].values
    regions, i = [], 0
    while i < len(active):
        if active[i]:
            j = i
            while j + 1 < len(active) and active[j + 1]: j += 1
            regions.append((t[i] - WIN_S / 2, t[j] + WIN_S / 2)); i = j + 1
        else: i += 1
    merged = []
    for a, b in regions:
        if merged and a - merged[-1][1] < MIN_GAP_S: merged[-1] = (merged[-1][0], b)
        else: merged.append((a, b))
    # refine using energy envelope
    frame, hop = int(0.01 * SR), int(0.005 * SR)
    rms = librosa.feature.rms(y=audio, frame_length=frame, hop_length=hop)[0]
    rt = librosa.frames_to_time(np.arange(len(rms)), sr=SR, hop_length=hop)
    noise_floor = np.percentile(rms, 30)
    events = []
    for a, b in merged:
        m = (rt >= max(a, 0)) & (rt <= min(b, DUR))
        if m.sum() == 0: continue
        # threshold relative to this region's own peak (coughs are much louder than breaths)
        thr_e = max(noise_floor * 4, 0.2 * rms[m].max())
        ids = np.where(m)[0]; pk_i = ids[np.argmax(rms[ids])]
        gap = int(0.08 / 0.005)                      # tolerate 80 ms dips inside one cough
        lo_i = pk_i
        while lo_i > ids[0]:
            if rms[max(lo_i - gap, ids[0]):lo_i].max() < thr_e: break
            lo_i -= 1
        hi_i = pk_i
        while hi_i < ids[-1]:
            if rms[hi_i:min(hi_i + gap, ids[-1])].max() < thr_e: break
            hi_i += 1
        s_, e_ = rt[lo_i], rt[hi_i]
        if e_ - s_ >= MIN_DUR_S: events.append((s_, e_))
    return events

events = detect_coughs(audio, P)
print(f"Detected {len(events)} cough event(s)")

if truth:
    tc = np.array([(a + b) / 2 for a, b in truth]); dc = np.array([(a + b) / 2 for a, b in events]) if events else np.array([])
    tp = sum(np.any(np.abs(dc - c) < 1.0) for c in tc) if len(dc) else 0
    fp = len(dc) - sum(np.any(np.abs(tc - d) < 1.0) for d in dc) if len(dc) else 0
    fn = len(tc) - tp
    prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
    print(f"Event-level  TP={tp}  FP={fp}  FN={fn}  |  precision={prec:.2f}  recall={rec:.2f}  F1={2*prec*rec/max(prec+rec,1e-9):.2f}")

## 9. Respiratory biomarkers
**Cough biomarkers:** count, rate (per hour), mean duration, peak intensity (dBFS), inter-cough interval, cough-epoch clustering.
**Breathing biomarker:** breathing rate (breaths/min) from the low-frequency amplitude envelope of the breath sounds (coughs masked out), estimated from a harmonic-summed spectrum (with a confidence score).

In [ ]:
def cough_biomarkers(audio, events):
    if not events: return dict(cough_count=0, cough_rate_per_hour=0.0)
    durs = np.array([b - a for a, b in events])
    peaks = [20 * np.log10(np.max(np.abs(audio[int(a * SR):int(b * SR) + 1])) + 1e-9) for a, b in events]
    onsets = np.array([a for a, _ in events]); ici = np.diff(onsets)
    # "epoch" = coughs <2 s apart (paroxysm / fit)
    epochs = 1 + int(np.sum(ici > 2.0)) if len(ici) else 1
    return dict(cough_count=len(events),
                cough_rate_per_hour=len(events) / (DUR / 3600),
                mean_cough_duration_s=float(durs.mean()),
                mean_peak_intensity_dBFS=float(np.mean(peaks)),
                mean_inter_cough_interval_s=float(ici.mean()) if len(ici) else np.nan,
                cough_epochs=epochs)

def breathing_rate(audio, events, lo_bpm=6, hi_bpm=40):
    x = bandpass(audio, SR, 150, 2000)
    hop = int(0.05 * SR); fs_env = SR / hop
    env = librosa.feature.rms(y=x, frame_length=int(0.2 * SR), hop_length=hop)[0]
    t = np.arange(len(env)) / fs_env
    # mask coughs (interpolate across them)
    mask = np.ones_like(env, bool)
    for a, b in events: mask &= ~((t > a - 0.3) & (t < b + 0.6))
    if mask.sum() > 10: env = np.interp(t, t[mask], env[mask])
    env = sps.sosfiltfilt(sps.butter(3, 1.2, fs=fs_env, output="sos"), env - env.mean())
    # (1) spectral estimate with harmonic summation.
    #     Inspiration AND expiration each create an envelope bump, so the strongest spectral line is often
    #     at 2x the breathing rate -> we score candidate f0 by P(f0) + P(2*f0) to recover the true rate.
    f, pxx = sps.welch(env, fs=fs_env, nperseg=min(len(env), int(40 * fs_env)), nfft=8192)
    cand = f[(f >= lo_bpm / 60) & (f <= hi_bpm / 60)]
    score = [np.interp(c, f, pxx) + np.interp(2 * c, f, pxx) for c in cand]
    bpm_spec = 60 * cand[int(np.argmax(score))]
    # (2) confidence: how dominant the chosen rhythm is vs. the rest of the breathing band (0..1)
    f0 = bpm_spec / 60; fb = (f >= lo_bpm / 60) & (f <= 2.2 * hi_bpm / 60)
    near = (np.abs(f - f0) < 0.15 * f0) | (np.abs(f - 2 * f0) < 0.15 * 2 * f0)
    conf = float(pxx[fb & near].sum() / (pxx[fb].sum() + 1e-12))
    pk, _ = sps.find_peaks(env, distance=int(60 / hi_bpm * fs_env), prominence=0.3 * env.std())
    return dict(breathing_rate_bpm_spectral=float(bpm_spec), breathing_confidence=round(conf, 2)), (t, env, pk)

cb = cough_biomarkers(audio, events)
bb, (tb, envb, pkb) = breathing_rate(audio, events)
report = {**cb, **bb}
rep_df = pd.DataFrame(report, index=["value"]).T.round(2)
display(rep_df)
if truth: print("Ground truth → coughs: %d | breathing: 16 bpm" % len(truth))

## 10. Visual report

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(13, 9), sharex=True)

# waveform + detections
ax[0].plot(np.arange(len(audio)) / SR, audio, lw=.5, color="gray")
for i, (a, b) in enumerate(events):
    ax[0].axvspan(a, b, color="red", alpha=.35, label="detected cough" if i == 0 else None)
if truth:
    for i, (a, b) in enumerate(truth):
        ax[0].plot([a, b], [1.05, 1.05], color="green", lw=3, label="ground truth" if i == 0 else None)
ax[0].set_ylabel("Amplitude"); ax[0].legend(loc="upper right"); ax[0].set_title("Cough detection")

# spectrogram
S = librosa.amplitude_to_db(np.abs(librosa.stft(audio, n_fft=1024, hop_length=256)), ref=np.max)
librosa.display.specshow(S, sr=SR, hop_length=256, x_axis="time", y_axis="log", ax=ax[1], cmap="magma")
ax[1].set_ylim(50, 4000); ax[1].set_title("Spectrogram (log-frequency)")

# breathing envelope
ax[2].plot(tb, envb, color="tab:blue"); ax[2].plot(tb[pkb], envb[pkb], "ro", ms=4)
ax[2].set_title(f"Breathing envelope — {report['breathing_rate_bpm_spectral']:.1f} breaths/min (spectral)")
ax[2].set_xlabel("Time (s)"); ax[2].set_ylabel("Envelope")
plt.tight_layout(); plt.show()

# zoom on one detected cough: waveform + mel-spectrogram
if events:
    a, b = events[0]; seg = audio[int(max(a - .3, 0) * SR):int((b + .3) * SR)]
    fig, ax = plt.subplots(1, 2, figsize=(11, 3))
    ax[0].plot(np.arange(len(seg)) / SR, seg, lw=.7); ax[0].set_title("First detected cough")
    M = librosa.power_to_db(librosa.feature.melspectrogram(y=seg, sr=SR, n_mels=64), ref=np.max)
    librosa.display.specshow(M, sr=SR, x_axis="time", y_axis="mel", ax=ax[1]); ax[1].set_title("Mel spectrogram")
    plt.tight_layout(); plt.show()
    display(Audio(seg, rate=SR))

## 11. Export results

In [ ]:
ev_df = pd.DataFrame([{"event": i + 1, "start_s": round(a, 2), "end_s": round(b, 2),
                       "duration_s": round(b - a, 2)} for i, (a, b) in enumerate(events)])
ev_df.to_csv("cough_events.csv", index=False)
rep_df.to_csv("respiratory_biomarkers_summary.csv")
display(ev_df)

import joblib; joblib.dump(clf, "resp_audio_classifier.joblib")
print("Saved: cough_events.csv, respiratory_biomarkers_summary.csv, resp_audio_classifier.joblib")

# Uncomment to download in Colab:
# from google.colab import files
# for f in ["cough_events.csv", "respiratory_biomarkers_summary.csv", "resp_audio_classifier.joblib"]: files.download(f)

## 12. Next steps / how to improve
| Goal | Idea |
|---|---|
| Better real-world accuracy | Train on **COUGHVID**, **Coswara**, **ICBHI** (lung sounds), **FSD50K**, **Virufy** |
| Deep learning | Replace Random Forest with a **CNN on log-mel spectrograms** or fine-tune **YAMNet / PANNs / AST** (`tensorflow_hub`) |
| Cough type | Add classes: wet / dry / whooping, wheeze, crackle, stridor |
| Longitudinal monitoring | Run on night-long recordings; track cough rate per hour & breathing rate trends |
| Robustness | Data augmentation (noise, time-stretch, gain, room impulse responses), speaker-independent splits |
| Clinical validity | Use patient-level cross-validation, report sensitivity/specificity, calibrate probabilities |

> ⚠️ **Disclaimer:** For research and education only. Not a medical device; not for diagnosis or treatment decisions.